# Call function with a easier way

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(override=True)

True

In [1]:
from typing import List
from pydantic import BaseModel, Field

# Instead of function_scheme dictionary
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport""" # docstring
    airport_code: str = Field(description="airport code to get weather for")    # parameter

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_function
 
weather_function = convert_to_openai_function(WeatherSearch)
weather_function

{'name': 'WeatherSearch',
 'description': 'Call this with an airport code to get the weather at that airport',
 'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [4]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="deepseek-chat", 
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL"))

response = model.invoke("what is the weather in SF today?", 
                        functions=[weather_function])
response

c:\Users\60410\Desktop\claude code\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='I don\'t have real-time access to current weather data, so I can\'t tell you the exact weather in San Francisco today. However, you can easily check it by:\n\n1. Searching "weather in San Francisco" on Google or your preferred search engine.\n2. Using a weather app like Weather.com, AccuWeather, or the default app on your phone.\n3. Asking a voice assistant like Siri or Alexa.\n\nAs of my knowledge cutoff, SF often has a mild, foggy, or overcast climate, especially in the summer months (the famous "Karl the Fog"). But for today\'s specifics, I\'d recommend checking a live source.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 12, 'total_tokens': 145, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 

In [6]:
model_with_function = model.bind_tools([weather_function])
response = model_with_function.invoke("what is the weather in sf?")
response

AIMessage(content='Let me check the weather for San Francisco. The airport code for San Francisco is SFO.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 302, 'total_tokens': 367, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 302}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b01c1e9c-1264-41cd-ba08-764dcd367d5e', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f32b1-0905-7dc0-9d63-2d3c9e5f9307-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_00_BLWSrPR0tmEZUtR6676i6741', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 302, 'output_tokens': 65, 'total_tokens': 367, 'input_token_details': {'cache_read': 0}, 'output_token_details':

In [8]:
response = model_with_function.invoke("Who are you")
response

AIMessage(content="Hello! I'm an AI assistant with access to tools that can help answer questions. I can help you search for weather information by airport code using the `WeatherSearch` tool.\n\nHow can I assist you today? Is there a particular airport's weather you'd like to know about?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 298, 'total_tokens': 355, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 42}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '7453d4de-5c63-4921-b4cb-7bf5296dafcb', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f32b9-179a-73b0-a961-1b9bd1570869-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'output_tokens': 57, 'total_tokens': 355, 'inpu

# Call function with chain

In [10]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

chain = prompt | model_with_function
response  = chain.invoke({"input": "what is the weather in sf?"})
response

AIMessage(content="Let me check the weather in San Francisco. I'll use the airport code for San Francisco (SFO).", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 308, 'total_tokens': 376, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 308}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'c6c67eb7-48b8-43af-bf42-2f8d730516e9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f32f8-b067-7021-8283-ae442e3ab8a9-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_00_VV116O4p3bifxct0UDdF9253', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 308, 'output_tokens': 68, 'total_tokens': 376, 'input_token_details': {'cache_read': 0}, 'output_token_de

In [17]:
arguments=response.tool_calls[0]['args']

arguments

{'airport_code': 'SFO'}

In [ ]:
# Multiple funcions       
class ProductPriceSearch(BaseModel):
    """Call this with product name  to get the price of product """
    product_name: str = Field(description="name of product to look up")
 
functions = [
    convert_to_openai_function(WeatherSearch),
    convert_to_openai_function(ProductPriceS
                               earch),
]
 
model_with_functions = model.bind_tools(functions)

model_with_functions.invoke("what is the weather in sf?")

AIMessage(content="Let me check the weather for San Francisco. I'll use the airport code for San Francisco (SFO).", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 369, 'total_tokens': 437, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 113}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ae9c5181-a5ca-4b2d-86fa-8eab2b784bed', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f3308-82b7-7c60-a52e-cf8c960f8a31-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_00_Zjsv4kmHvUyOut3PO1X51262', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 369, 'output_tokens': 68, 'total_tokens': 437, 'input_token_details': {'cache_read': 256}, 'output_t

In [ ]:
# LLM can judge which function to use based on the context
model_with_functions.invoke("what are the price of iphone 16 pro")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 373, 'total_tokens': 422, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 117}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '51cb3f39-4b46-447a-8bdb-5ad14cb43288', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f3309-1ada-7ed0-a380-179608e6ce18-0', tool_calls=[{'name': 'ProductPriceSearch', 'args': {'product_name': 'iphone 16 pro'}, 'id': 'call_00_cvKrUcZwr6kBqpZUUu821561', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 373, 'output_tokens': 49, 'total_tokens': 422, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}})